# Train GCIDM on OGBCubeDR (RunPod)

Trains the **goal-conditioned inverse-dynamics world model** on
`cube_quadruple_dr_expert` (5000 `OGBCubeDR-v0` episodes, see `Run.md`). Flow:
config → apply → torch check → install → download → sanity check →
**shortcut-ceiling pre-test** → train.

GCIDM is SMWM with the single-step IDM (`(z_t, z_{t+1}) -> a_t`) replaced by a
**goal-conditioned plan head** (`stable_worldmodel.wm.gcidm.module.GoalPolicyHead`):

```
pi( z_{t-2}, z_{t-1}, z_t ,  z_goal ) -> a_t .. a_{t+4}   (5 action blocks)
     └── predictor's context ──┘   └ frame t + 25 env steps ┘
```

trained with

```
L = pred_loss
    + loss.policy.weight     * policy_loss      # MSE on mu, shapes the encoder
    + loss.policy.std_weight * policy_std_loss  # NLL on a DETACHED mu
    + loss.sigreg.weight     * sigreg_loss      # kept, 0.0 by default
```

**Why the change.** SMWM's one-step IDM only needs whatever differs between two
adjacent frames, which on this task is almost entirely the gripper — which is
exactly what the DR probing found it recovered. Over 25 env steps the plan has
to route around and grasp a cube, so the head cannot emit it without encoding
where that cube is. The head also does double duty: at planning time it
warm-starts CEM (see `plan_gcidm_ogbcubedr.ipynb`).

**Why `policy_std_loss` regresses a detached mean.** Under a joint
heteroscedastic NLL the weight on the mean's error is `1/sigma^2`, and `sigma`
is a free parameter the head controls: it could shrink `sigma` on easy samples
and inflate it on the hard, task-relevant ones, down-weighting exactly the
gradients we want reaching the encoder — and with `sigreg.weight=0` and no EMA
target, `policy_loss` is the *only* anti-collapse term. Detaching confines the
NLL gradient to the sigma head's own weights.

`scripts/train/config/gcidm.yaml` keeps LeWM's architecture
(`encoder_scale=small`, `embed_dim=384`, `wm.history_size=3`) and differs from
`smwm.yaml` in:

| Key | `smwm.yaml` | `gcidm.yaml` | Why |
|---|---|---|---|
| `model.policy_head` | (`inverse_model`) | `GoalPolicyHead` | the 5-block plan head, ~1.97M params vs the predictor's ~24M |
| `wm.goal_horizon` | — | `5` | blocks to the goal frame = 25 env steps |
| `loss.policy.weight` | (`loss.inverse.weight`) | `1.0` | the new shaping term |
| `loss.policy.std_weight` | — | `1.0` | calibrates sigma for the CEM warm start |
| `data` | `ogb_cube_quadruple_dr` | `ogb_cube_quadruple_dr_goal` | `num_steps` 4 → 8, so the clip reaches the goal frame |

Frames 4–6 of the 8-frame clip feed no loss, so only `[0,1,2,3,7]` reach the
ViT: 2× SMWM's clip span but ~1.25× its encoder cost.

Prerequisite: dataset already pushed to HF via `upload_to_hf.sh` (repo root).


## 1. Config

Edit the values below. `STABLEWM_HOME=/workspace` puts the dataset at `/workspace/datasets/ogbench/...`. `HF_TOKEN`/`WANDB_API_KEY` come from pod env vars if set, else paste them in.

In [ ]:
import os

# --- repo ---
REPO_ROOT = '/workspace/stable-worldmodel'          # ← edit if you cloned it elsewhere

# --- storage (network volume) ---
STABLEWM_HOME = '/workspace'                        # datasets/, checkpoints/ land directly here
SPT_CACHE_DIR = '/workspace/cache/stable-pretraining'  # Lightning .ckpt files (see note below)

# --- Hugging Face ---
HF_TOKEN = os.environ.get('HF_TOKEN', '')                          # ← paste here if not set as a pod env var
HF_REPO_ID = 'quastAI/ogbench-cube-quadruple-domain-randomized-expert'  # ← edit me

# --- Weights & Biases ---
WANDB_API_KEY = os.environ.get('WANDB_API_KEY', '')  # ← paste here if not set as a pod env var
WANDB_ENTITY = 'julian-quast-8-technical-university-of-berlin'                 # ← edit me
WANDB_PROJECT = 'ogbcubedr-gcidm'                     # ← edit me if you want a different project name

# --- training run naming ---
OUTPUT_MODEL_NAME = 'gcidm_q4_dr'

# --- derived, don't edit ---
DATASET_DIR = os.path.join(STABLEWM_HOME, 'datasets', 'ogbench', 'cube_quadruple_dr_expert.lance')

## 2. Apply config

In [ ]:
os.environ['STABLEWM_HOME'] = STABLEWM_HOME
os.environ['SPT_CACHE_DIR'] = SPT_CACHE_DIR
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['WANDB_API_KEY'] = WANDB_API_KEY

os.makedirs(STABLEWM_HOME, exist_ok=True)
os.makedirs(SPT_CACHE_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

os.chdir(REPO_ROOT)  # os.chdir (not `%cd`) so it's identical whether run fresh or after a kernel restart

print('cwd           =', os.getcwd())
print('STABLEWM_HOME =', os.environ['STABLEWM_HOME'])
print('SPT_CACHE_DIR =', os.environ['SPT_CACHE_DIR'])
print('DATASET_DIR   =', DATASET_DIR)
!df -h /workspace

## 3. Torch ≥ 2.5

`transformers` needs `torch>=2.5`; some pods ship 2.4.1. Upgrades torch+torchvision+torchaudio together. **Restart the kernel if it upgrades**, then re-run cells 1–2.

In [ ]:
import torch
print('torch before:', torch.__version__)

if tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (2, 5):
    !pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
    print('Upgraded — RESTART THE KERNEL now, then re-run cells 1-2 before continuing.')
else:
    print('torch already >= 2.5, nothing to do.')

## 4. Install dependencies

In [ ]:
%pip install -q -e '.[train,format]' wandb huggingface_hub

## 5. Download dataset

Uses `HF_TOKEN` from the config cell. Lands at `$STABLEWM_HOME/datasets/ogbench/cube_quadruple_dr_expert.lance/`. Safe to re-run.

In [ ]:
!hf download "$HF_REPO_ID" --repo-type dataset --local-dir "$DATASET_DIR"

## 6. W&B login

In [ ]:
import wandb

wandb.login(key=WANDB_API_KEY)

## 7. Sanity check

Expect 5000 episodes × 401 steps, and a visible GPU.

In [ ]:
import lance

ds = lance.dataset(DATASET_DIR)
ep = ds.to_table(columns=['episode_idx']).column('episode_idx').to_numpy()
print(f'rows: {ds.count_rows():,}')
print(f'episodes: {ep.max() - ep.min() + 1}')

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU visible — gcidm.yaml requires accelerator: gpu'
print(torch.cuda.get_device_name(0))

## 7.5 Shortcut-ceiling pre-test — run this before training

**This gates the run.** GCIDM's whole bet is that predicting the plan forces
cube state into the latent. There is a shortcut that would defeat it: the
oracle's action is literally an effector delta
(`plan[i] - proprio/effector_pos`), so the *sum* of the plan is computable from
gripper features alone, with no idea where any cube is. Only the path curvature
(approach above the cube, descend, grasp, lift, clearance waypoint) and the
grasp timing genuinely need cube state.

`scripts/probe/shortcut_ceiling.py` measures how much cube state is worth,
using **ground-truth state** instead of any encoder — so it needs no checkpoint,
runs on CPU in minutes, and answers the question *before* a 100-epoch run:

| Feature set | Columns | Meaning |
|---|---|---|
| `proprio` | every `proprio/*` (19 dims) | **shortcut ceiling** — best R² using the robot only |
| `proprio_cubes` | the same **plus** ground-truth block poses (51 dims) | **information ceiling** — a strict superset, so the difference is exactly what cube state adds |

Read the gap: **< ~0.05 → NO-GO** at this `goal_horizon` (no encoder can be
pushed to represent cube state by this objective, so train the long-horizon
variant instead); **large → GO**, and record both numbers as the axis for
`policy_loss` in §8.

In [ ]:
!python scripts/probe/shortcut_ceiling.py \
    --goal-horizon 5 --frameskip 5 --context-frames 3 \
    --out "$STABLEWM_HOME/runs/shortcut_ceiling_h5"

And the sweep that picks the long-horizon offset empirically, should the
`goal_horizon=5` gap come back too small to work with — the widest-gap offset
is where a shaping head would actually be forced to encode cubes:

In [ ]:
for H in [5, 10, 20]:
    !python scripts/probe/shortcut_ceiling.py \
        --goal-horizon {H} --frameskip 5 --context-frames 3 \
        --out "$STABLEWM_HOME/runs/shortcut_ceiling_h{H}"

## 8. Train

Smoke test first, then the full run. The smoke run deliberately keeps the
shipped `loader.batch_size=256` so that an out-of-memory pod fails here, in one
epoch, rather than an hour into the real run. If it OOMs, either drop to
`loader.batch_size=128 optimizer.lr=5e-5` or keep the effective batch with
`loader.batch_size=128 +trainer.accumulate_grad_batches=2`.

In [ ]:
!python scripts/train/gcidm.py data=ogb_cube_quadruple_dr_goal \
    output_model_name="${OUTPUT_MODEL_NAME}_smoke" \
    trainer.max_epochs=1 \
    loader.num_workers=2 \
    wandb.enabled=false \
    hydra.run.dir=/tmp/gcidm_smoke

Full run — launched detached (survives closing the browser/kernel; only dies if the pod stops):

In [ ]:
import subprocess

log_path = os.path.join(STABLEWM_HOME, 'logs', f'{OUTPUT_MODEL_NAME}.log')
os.makedirs(os.path.dirname(log_path), exist_ok=True)

cmd = [
    'python', 'scripts/train/gcidm.py',
    'data=ogb_cube_quadruple_dr_goal',
    f'output_model_name={OUTPUT_MODEL_NAME}',
    'wandb.enabled=true',
    f'wandb.config.entity={WANDB_ENTITY}',
    f'wandb.config.project={WANDB_PROJECT}',
]

with open(log_path, 'w') as f:
    proc = subprocess.Popen(
        cmd, stdout=f, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        start_new_session=True,  # detaches from this kernel's process group/session
    )

print('Started PID', proc.pid)
print('Log:', log_path)

Check progress any time (also visible live on the W&B dashboard):

In [ ]:
!tail -n 40 "$log_path"

### What to watch

`policy_loss` is the number this model adds, and with `loss.sigreg.weight=0` it
is also the **only** thing preventing representational collapse — `pred_loss`
alone is minimised at zero by a constant encoder, and there is no EMA target
network.

Because the action column is z-scored, the 125-dim label has per-dim variance
≈1, so `policy_loss` reads directly as explained variance:

```
R2 ~= 1 - policy_loss
```

Compare that against the two ceilings from §7.5:

- `policy_loss` at the **shortcut ceiling** → the head is solving the task from
  gripper features alone and the latent gained nothing over SMWM.
- `policy_loss` meaningfully **below** it → cube information is being used, and
  the margin says how much.

Other things to watch:

- `pred_loss` falling **and** `policy_loss` falling → healthy.
- `pred_loss` diving while `policy_loss` plateaus high → collapse. Raise
  `loss.policy.weight`, or put SIGReg back with `loss.sigreg.weight=0.09` (that
  value was tuned at batch 128, and the SIGReg statistic scales with batch size,
  so re-tune it if you keep `batch_size=256`).
- `policy_std_loss` going very negative → `sigma` is collapsing toward the
  clamp at `exp(-5)`. Harmless for training (the term is detached from
  everything else) but it makes the CEM warm start over-confident; the solver's
  `var_floor` is the backstop.

`probe_gcidm_ogbcubedr.ipynb` is the direct test of whether the latent kept the
state: a collapsed encoder scores at the constant-predictor baseline on every
target, and the specific claim here is that **cube** positions become
decodable where SMWM recovered only gripper state.

To train the plain LeWM objective from this same script — a strict A/B — zero
the new term and restore SIGReg:

```bash
python scripts/train/gcidm.py data=ogb_cube_quadruple_dr_goal \
    loss.policy.weight=0.0 loss.sigreg.weight=0.09 \
    loader.batch_size=128 optimizer.lr=5e-5
```


Checkpoints: `$STABLEWM_HOME/checkpoints/$OUTPUT_MODEL_NAME/weights_epoch_N.pt` (used by `eval_wm.py`). Lightning's `.ckpt`s go to `$SPT_CACHE_DIR/runs/.../checkpoints/` — not used by eval. See `Run.md` §6–7.